# hidden cohort の中身 — 属性の構成と cohort × class の class weight

iterative の各 stage は、属性を使わずに作った 10 個の hidden cohort で GroupDRO を回す。cohort の中身が既知の属性と
どれだけ重なるかで、GroupDRO と class weight が属性群に何をしているかの読み方が変わる。この notebook は
**学習の前に決まる cohort artifact の性質**だけを見る。epoch ごとの推移は
[`subgroup_training_curves.ipynb`](subgroup_training_curves.ipynb) で扱った。

問いは 3 つある。

1. cohort は sex・age group・race とどれだけ重なるか。
2. stage の loss が使う cohort × class の class weight（`weighting=inverse`）は、cohort 間でどれだけ違い、何と連動するか。
3. `q` の高い cohort と低い cohort は、race・sex・age・撮影方向（view position）で見てどう違うか。

cohort の構成は train split の割り当てで見る。class weight も `q` も train の上で決まるためである。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.style
import numpy as np
import pandas as pd
import rootutils

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from analysis.common.paths import STYLE_SHEET, run_dir, split_csv  # noqa: E402
from analysis.common.run_artifacts import read_config  # noqa: E402
from analysis.iterative_probe._shared import collect_epoch_rows, rank_cohorts_by_q, read_condition  # noqa: E402

matplotlib.style.use(STYLE_SHEET)
pd.set_option("display.max_rows", 120)

## 対象 run と定数

run は条件順（fc 1e-3 → fc 1e-2 → stage4+fc 1e-3 → stage4+fc 1e-2）に並べる。cohort は stage ごとに引き直すので、
stage01 と stage02 の同じ番号の cohort は別の集団である。

race の符号は split CSV のもので、0 = White、2 = Asian、3 = Black（`groups.RACE_LABELS`）。残りの 3 コード（1・4・5）は
名前の対応が repo に無いので、割合の表では「その他」にまとめ、ヒストグラムではコードのまま出す。

撮影方向は `frontal_lateral`（0 = Frontal、1 = Lateral）と `ap_pa`（0 = AP、1 = PA）を 1 つにまとめ、AP / PA / Lateral の 3 値にする。
Lateral の行は `ap_pa` が欠損している。Frontal なのに `ap_pa` が欠損している 12 行は別の値として残す。

In [ ]:
STUDY = "iterative_probe"
ITERATIVE_RUN_IDS = [
    "20260924T112035Z-spatial-lora-iterative-chexpert-fc-s42-b81a",
    "20260924T112045Z-spatial-lora-iterative-chexpert-fc-s42-4602",
    "20260924T112034Z-spatial-lora-iterative-chexpert-stage4-fc-s42-1caa",
    "20260924T112043Z-spatial-lora-iterative-chexpert-stage4-fc-s42-c755",
]
COHORT_STAGES = ["stage01", "stage02"]
RACE_CODES = {"white": 0, "asian": 2, "black": 3}
RACE_NAMES = {0: "White", 1: "code 1", 2: "Asian", 3: "Black", 4: "code 4", 5: "code 5"}
SEX_NAMES = {0: "male", 1: "female"}
VIEW_NAMES = ["AP", "PA", "Lateral", "Frontal (AP/PA missing)"]
PICK = 2
PURE_THRESHOLD = 0.05

RESULTS = ROOT / "analysis" / STUDY / "results"
FIGURES = ROOT / "analysis" / STUDY / "figures"
RESULTS.mkdir(parents=True, exist_ok=True)

## 入力の読み込み

- cohort の割り当て: `artifacts/cohorts/cohortNN/assignments.parquet`（`split`・`image`・`group_id`）。stage01 は `cohort01`、stage02 は `cohort02`。
- 属性と正解: train の split CSV を `image` でつなぐ。`frontal_lateral` は train に欠損が無い。
- class weight: `stages/<stage>/config.yaml` の `model.loss_fn.class_weight`（`[cohort, class]`）。

In [ ]:
train_frame = pd.read_csv(split_csv("chexpert", "train"))[
    [
        "image",
        "target",
        "age",
        "age_missing",
        "sex",
        "sex_missing",
        "race",
        "race_missing",
        "frontal_lateral",
        "frontal_lateral_missing",
        "ap_pa",
        "ap_pa_missing",
    ]
]
frontal = train_frame["frontal_lateral"] == 0
ap_pa_known = ~train_frame["ap_pa_missing"].astype(bool)
train_frame["view"] = np.select(
    [
        ~frontal,
        frontal & ap_pa_known & (train_frame["ap_pa"] == 0),
        frontal & ap_pa_known & (train_frame["ap_pa"] == 1),
    ],
    ["Lateral", "AP", "PA"],
    default="Frontal (AP/PA missing)",
)
# CheXpert は欠損の race に White と同じ 0 が入っているので、フラグで `missing` に分ける。
train_frame["race_label"] = (
    train_frame["race"].map(RACE_NAMES).where(~train_frame["race_missing"].astype(bool), "missing")
)
train_frame["sex_label"] = train_frame["sex"].map(SEX_NAMES)


def read_assignments(run_id: str, stage: str) -> pd.DataFrame:
    cohort_name = f"cohort{stage.removeprefix('stage')}"
    path = run_dir(STUDY, run_id) / "artifacts" / "cohorts" / cohort_name / "assignments.parquet"
    assignments = pd.read_parquet(path)
    return assignments[assignments["split"] == "train"].merge(train_frame, on="image", validate="one_to_one")


assignments = {
    (run_id, stage): read_assignments(run_id, stage) for run_id in ITERATIVE_RUN_IDS for stage in COHORT_STAGES
}
{key[1] + " / " + key[0][-4:]: len(frame) for key, frame in assignments.items()}

同じ変調範囲の 2 run は warmup が一致するので、stage01 の cohort も一致するはずである。割り当てそのもので確かめる。

In [ ]:
def same_cohorts(first: str, second: str, stage: str) -> bool:
    left = assignments[(first, stage)].set_index("image")["group_id"]
    right = assignments[(second, stage)].set_index("image")["group_id"]
    return bool(left.sort_index().equals(right.sort_index()))


{
    "fc stage01": same_cohorts(ITERATIVE_RUN_IDS[0], ITERATIVE_RUN_IDS[1], "stage01"),
    "stage4+fc stage01": same_cohorts(ITERATIVE_RUN_IDS[2], ITERATIVE_RUN_IDS[3], "stage01"),
    "fc stage02": same_cohorts(ITERATIVE_RUN_IDS[0], ITERATIVE_RUN_IDS[1], "stage02"),
}

## 1. cohort の属性の構成

cohort ごとに、65 歳以上の割合・女性の割合（`sex` = 1）・race の各割合・撮影方向の割合・陽性率を出す。割合の分母は、
その属性が欠損していない行である（撮影方向は全行）。race は欠損が 1 割あり、欠損の偏りそのものも cohort の性質になりうるので、
`race_missing`（全行に占める欠損の割合）を別に出す。

In [ ]:
def share(values: pd.Series, missing: pd.Series, code: int) -> float:
    observed = values[~missing.astype(bool)]
    return float((observed == code).mean())


def summarize_cohort(frame: pd.DataFrame) -> pd.Series:
    old = (frame["age"] >= 65).astype(int)
    row = {
        "n": len(frame),
        "positive_rate": frame["target"].mean(),
        "age_65_plus": share(old, frame["age_missing"] | frame["age"].isna(), 1),
        "female": share(frame["sex"], frame["sex_missing"], 1),
        "race_missing": frame["race_missing"].astype(bool).mean(),
    }
    for name, code in RACE_CODES.items():
        row[name] = share(frame["race"], frame["race_missing"], code)
    for view in ["AP", "PA", "Lateral"]:
        row[view.lower()] = float((frame["view"] == view).mean())
    return pd.Series(row)


composition_rows = []
for (run_id, stage), frame in assignments.items():
    summary = frame.groupby("group_id").apply(summarize_cohort, include_groups=False)
    class_weight = read_config(run_dir(STUDY, run_id) / "stages" / stage / "config.yaml")["model"]["loss_fn"][
        "class_weight"
    ]
    summary["negative_weight"] = [weights[0] for weights in class_weight]
    summary["positive_weight"] = [weights[1] for weights in class_weight]
    condition = read_condition(run_dir(STUDY, run_id))["condition"]
    composition_rows.append(summary.reset_index().assign(run_id=run_id, condition=condition, stage=stage))

composition = pd.concat(composition_rows, ignore_index=True).rename(columns={"group_id": "cohort"})
composition["cohort"] = composition["cohort"].astype(int)
composition.to_csv(RESULTS / "cohort_composition.csv", index=False)
composition[composition["condition"] == "fc / 0.001"].set_index(["stage", "cohort"]).drop(
    columns=["run_id", "condition"]
).round(3)

train 全体の割合を並べておく。cohort の割合がこれからどれだけ離れているかを見る基準になる。

In [ ]:
summarize_cohort(train_frame).round(3)

cohort がどこまで属性で割り切れるかを、全 run × stage でまとめる。`pure_*` は、その属性の割合が 0.05 未満か 0.95 を
超える（ほぼ片方の値だけからなる）cohort の数。race は White の割合で見る。撮影方向は 3 値なので、`pure_view` は最多の方向が
0.95 を超える cohort の数、`no_ap` は AP が 0.05 未満の cohort（PA と Lateral だけからなる）の数とする。
`race_missing_over_half` は race の欠損が半分を超える cohort の数。

In [ ]:
def count_pure(values: pd.Series) -> int:
    return int(((values < PURE_THRESHOLD) | (values > 1 - PURE_THRESHOLD)).sum())


composition["view_major_share"] = composition[["ap", "pa", "lateral"]].max(axis=1)
purity = composition.groupby(["condition", "stage"], sort=False).agg(
    pure_sex=("female", count_pure),
    pure_age=("age_65_plus", count_pure),
    pure_white=("white", count_pure),
    pure_view=("view_major_share", lambda values: int((values > 1 - PURE_THRESHOLD).sum())),
    no_ap=("ap", lambda values: int((values < PURE_THRESHOLD).sum())),
    race_missing_over_half=("race_missing", lambda values: int((values > 0.5).sum())),
    white_min=("white", "min"),
    white_max=("white", "max"),
)
purity.round(3)

## 2. cohort × class の class weight

`weighting=inverse` は cohort ごとに陽性率の逆数で重みを付けるので、陽性の重みは陽性率でほぼ決まる。ここで見たいのは、
その陽性率が属性とどう連動するか、つまり **class weight が実質的にどの属性ごとに閾値を動かしているか**である。
cohort ごとの陽性の重みと、65 歳以上の割合・女性の割合・White の割合との Spearman 順位相関を run × stage ごとに出す。

In [ ]:
def weight_relation(part: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "positive_weight_ratio": part["positive_weight"].max() / part["positive_weight"].min(),
            "rho(weight, positive_rate)": part["positive_weight"].corr(part["positive_rate"], method="spearman"),
            "rho(weight, age_65_plus)": part["positive_weight"].corr(part["age_65_plus"], method="spearman"),
            "rho(weight, female)": part["positive_weight"].corr(part["female"], method="spearman"),
            "rho(weight, white)": part["positive_weight"].corr(part["white"], method="spearman"),
        }
    )


weight_relations = pd.DataFrame(
    {key: weight_relation(part) for key, part in composition.groupby(["condition", "stage"], sort=False)}
).T
weight_relations.round(3)

属性ごとの陽性率も確かめる。cohort の陽性率の差が属性の陽性率の差から来ているなら、train 全体でも同じ向きの差がある。
撮影方向も並べる（3 節で、`q` の高い cohort が撮影方向で割れていることを見る）。

In [ ]:
train_observed = train_frame[~train_frame["age_missing"].astype(bool) & train_frame["age"].notna()]
positive_rates = {
    "age<65": train_observed.loc[train_observed["age"] < 65, "target"].mean(),
    "age>=65": train_observed.loc[train_observed["age"] >= 65, "target"].mean(),
    "male": train_frame.loc[train_frame["sex"] == 0, "target"].mean(),
    "female": train_frame.loc[train_frame["sex"] == 1, "target"].mean(),
}
positive_rates |= train_frame.groupby("view")["target"].mean().to_dict()
pd.Series(positive_rates, name="positive_rate").round(3)

## 3. `q` の高い cohort と低い cohort の構成

[`subgroup_training_curves.ipynb`](subgroup_training_curves.ipynb) の 3 節と同じ規則（`_shared.rank_cohorts_by_q`）で、
stage ごとに **stage 内平均の `q` で上位 2 cohort と下位 2 cohort** を選ぶ。その 4 cohort の race・sex・age・撮影方向の
分布を、train 全体の分布と並べる。

step size 1e-3 は `q` がほぼ一様なので、上位と下位の違いは小さい。1e-2 との対照として並べる。stage01 の cohort は同じ変調範囲の
2 run で同じ集団なので、選ばれる cohort が重なることがある。

In [ ]:
q_rank = pd.concat(
    [rank_cohorts_by_q(pd.DataFrame(collect_epoch_rows(run_dir(STUDY, run_id))), PICK) for run_id in ITERATIVE_RUN_IDS],
    ignore_index=True,
)
picked = q_rank[q_rank["pick"] != "middle"].copy()
# 高い群は q の大きい順に 1, 2、低い群は q の小さい順に 1, 2 を振り、図の色を決める。
picked["order"] = picked.groupby(["run_id", "stage", "pick"])["q_mean"].rank(ascending=False).astype(int)
low = picked["pick"] == "low"
picked.loc[low, "order"] = picked[low].groupby(["run_id", "stage"])["q_mean"].rank().astype(int)

picked_composition = picked.merge(composition, on=["run_id", "condition", "stage", "cohort"], validate="one_to_one")
picked_composition.to_csv(RESULTS / "cohort_q_picked_composition.csv", index=False)
picked_composition[
    [
        "condition",
        "stage",
        "cohort",
        "pick",
        "q_mean",
        "n",
        "positive_rate",
        "positive_weight",
        "age_65_plus",
        "female",
        "white",
        "asian",
        "black",
        "ap",
        "pa",
        "lateral",
    ]
].round(3)

図は条件ごとに 1 枚。**行が cohort、列が属性**で、1 つの subplot は 1 つの cohort の 1 つの属性の分布だけを描く。

- 行は stage01 の 4 cohort、stage02 の 4 cohort の順。stage の中では `q` の大きい順（上位 1・上位 2・下位 2・下位 1）に並べる。
- 棒の色は [`subgroup_training_curves.ipynb`](subgroup_training_curves.ipynb) の `cohort_q_picked_curves.png` と揃え、
  上位 2 cohort を赤系、下位 2 cohort を青系にする。
- 灰色の破線の枠は、同じ stage の train 全体の分布。cohort の棒がここからどれだけ離れているかを見る。
- 縦軸は cohort の中での割合。age は 5 歳刻みで、点線が 65 歳（age group の境界）。

In [ ]:
PICK_COLOR = {("high", 1): "#C0392B", ("high", 2): "#E67E22", ("low", 1): "#1F618D", ("low", 2): "#48C9B0"}
PICK_ORDER = [("high", 1), ("high", 2), ("low", 2), ("low", 1)]
TRAIN_EDGE = "#8a8a86"
AGE_BINS = np.arange(0, 95, 5)
ATTRIBUTE_PANELS = {
    "race_label": ("race", [*RACE_NAMES.values(), "missing"]),
    "sex_label": ("sex", list(SEX_NAMES.values())),
    "age": ("age", None),
    "view": ("view position", VIEW_NAMES),
}
VIEW_TICKS = {"Frontal (AP/PA missing)": "Frontal\n(no AP/PA)"}


def draw_attribute(axis, cohort: pd.DataFrame, overall: pd.DataFrame, column: str, categories, color: str) -> None:
    if categories is None:
        for frame, style in [
            (cohort, {"color": color}),
            (overall, {"histtype": "step", "color": TRAIN_EDGE, "linestyle": "--", "linewidth": 1.2}),
        ]:
            axis.hist(frame[column], bins=AGE_BINS, weights=np.full(len(frame), 1 / len(frame)), **style)
        axis.axvline(65, color="#555555", linestyle=":", linewidth=1)
        return
    positions = np.arange(len(categories))
    for frame, style in [
        (cohort, {"color": color}),
        (overall, {"fill": False, "edgecolor": TRAIN_EDGE, "linestyle": "--", "linewidth": 1.2}),
    ]:
        shares = frame[column].value_counts(normalize=True).reindex(categories, fill_value=0)
        axis.bar(positions, shares, 0.7, **style)
    axis.set_xticks(positions, [VIEW_TICKS.get(name, name) for name in categories], rotation=45, ha="right")


def plot_picked_composition(run_id: str) -> None:
    rows = picked[picked["run_id"] == run_id].set_index(["stage", "pick", "order"])
    figure, axes = plt.subplots(
        len(COHORT_STAGES) * len(PICK_ORDER), len(ATTRIBUTE_PANELS), figsize=(15, 19), sharex="col", sharey="col"
    )
    for stage_index, stage in enumerate(COHORT_STAGES):
        overall = assignments[(run_id, stage)]
        for pick_index, key in enumerate(PICK_ORDER):
            row = stage_index * len(PICK_ORDER) + pick_index
            cohort = rows.loc[(stage, *key)]
            members = overall[overall["group_id"] == cohort["cohort"]]
            for col, (column, (title, categories)) in enumerate(ATTRIBUTE_PANELS.items()):
                draw_attribute(axes[row, col], members, overall, column, categories, PICK_COLOR[key])
                if row == 0:
                    axes[row, col].set_title(title)
            axes[row, 0].set_ylabel(
                f"{stage} {key[0]} {key[1]}\ncohort {int(cohort['cohort']):02d}\n"
                f"q={cohort['q_mean']:.3f}, n={len(members)}",
                fontsize=8,
            )
    condition = read_condition(run_dir(STUDY, run_id))["condition"]
    figure.suptitle(f"{condition}: composition of high / low q cohorts (train, share in cohort)", y=1.0)
    figure.tight_layout()
    slug = condition.replace(" / ", "_").replace("+", "-")
    figure.savefig(FIGURES / f"cohort_q_picked_composition_{slug}.png", dpi=150, bbox_inches="tight")
    plt.show()

条件順（fc 1e-3 → fc 1e-2 → stage4+fc 1e-3 → stage4+fc 1e-2）に 4 枚描く。race の欠損（`race_missing`）は `missing`
として別の値にしている（入力の読み込みの節）。

In [ ]:
for run_id in ITERATIVE_RUN_IDS:
    plot_picked_composition(run_id)

上位 2 cohort と下位 2 cohort の割合を平均して並べる。`high − low` が正なら、`q` の高い側にその属性が多い。

In [ ]:
SHARE_COLUMNS = [
    "age_65_plus",
    "female",
    "race_missing",
    "white",
    "asian",
    "black",
    "ap",
    "pa",
    "lateral",
    "positive_rate",
]
pick_mean = picked_composition.groupby(["condition", "stage", "pick"], sort=False)[SHARE_COLUMNS].mean().unstack("pick")
pick_difference = pick_mean.xs("high", axis=1, level="pick") - pick_mean.xs("low", axis=1, level="pick")
pick_difference.round(3)

## まとめ

数値の正本は `results/cohort_composition.csv` と `results/cohort_q_picked_composition.csv`。読み取りの記録は
[reports/cohort_composition.md](reports/cohort_composition.md) に残す。

**観察**

- **cohort は sex・撮影方向・age で割れている。**
  - sex: stage01 は 10 cohort のうち片方の sex だけからなるものが fc で 10 個、stage4+fc で 9 個。stage02 では 5〜7 個に減る。
  - 撮影方向: 1 つの方向だけからなる cohort が stage ごとに 3〜5 個ある。AP を含まない（PA と Lateral だけの）cohort は 2〜3 個。
  - age group: 片方だけからなる cohort が 3〜5 個。
- **race では割れないが、race の欠損が偏る。** White の割合（欠損を除く）は 0.18〜0.79 で、0.05 / 0.95 を超える cohort は無い。
  一方、race の欠損が半分を超える cohort が stage ごとに 1〜2 個ある（train 全体の欠損は 0.11）。fc の stage01 の cohort 03 と 07
  は欠損が 0.66 / 0.58 で、残りも大半が code 1 である。
- **陽性の class weight は 65 歳以上の割合と強く連動する。** 順位相関は +0.68〜+0.86、sex とはほぼ無相関（−0.13〜+0.15）。
  train 全体の陽性率は 65 歳未満 0.137、65 歳以上 0.054、男性 0.100、女性 0.102。撮影方向では AP 0.071、PA 0.188、Lateral 0.168。
- **`q` の上位 2 cohort は、ほぼ AP を含まない cohort である。** 16 個（4 run × 2 stage × 2）のうち 14 個は AP が 0.05 未満。
  例外の 2 個は stage4+fc の stage01 の同じ cohort 00（65 歳以上の女性、AP 0.79）で、2 run で共通の集団である。
  下位 2 cohort は 16 個すべてで AP が 0.62〜1.0。
- **上位と下位は sex と race の欠損でも違う。** 上位 − 下位の差は、女性の割合で 8 通りすべて負（−0.15〜−1.00）、race の欠損の
  割合で 8 通りすべて負（−0.06〜−0.96）。race の欠損が過半の cohort は延べ 13 個（run × stage で数えるので、stage01 の cohort は 2 回数える）あり、
  そのうち 8 個が下位 2 に入り、上位 2 には 1 個も入らない。
- **陽性率では上位と下位が分かれない。** 上位 − 下位の差は −0.03〜+0.10。
- 上の傾向は step size 1e-3（`q` がほぼ一様）でも 1e-2 と同じ向きに出る。stage01 の cohort は同じ変調範囲の 2 run で割り当てが一致する。

**解釈候補**

- cohort × class の class weight は、実質的に **age group ごとに陽性の重みを変えている**。これは age group ごとに判定の閾値を
  合わせ直すことに近く、[subgroup_training_curves](subgroup_training_curves.ipynb) で見た「stage に入ると age group の Eopp が
  `q` と無関係に下がる」ことと合う。
- **GroupDRO が重みを寄せているのは、撮影方向の少数派（PA・Lateral）の cohort である。** train の 72% が AP なので、AP を含まない
  cohort は ERM で学習の割合が小さく、train の損失が高く残りやすい。`q` の高い cohort が stage の間ずっと弱いまま（subgroup
  notebook の 3 節）であることとも合う。撮影方向は公平性の評価軸ではない（撮影プロトコル）ので、この設定の GroupDRO は
  公平性の評価軸の群ではなく、撮影条件の群の損失を下げにいっていることになる。
- 上位に男性が多いのは、AP を含まない cohort が男性側で切り出されやすいことの表れかもしれない。sex そのものが `q` を
  決めているのかは、この表からは分けられない。
- race の欠損が偏った cohort があるのは、画像の側に欠損と結び付く特徴（撮影時期や撮影元など）があることを示唆する。
- hidden cohort が sex × 撮影方向 × age を拾っているので、この設定の hidden group は既知の属性と撮影条件の近似になっている。
  未知の群を見つけるという目的に照らしてこれでよいかは、別に判断が要る。